# Import Library

In [1]:
import os
import pandas as pd
import io
from google import genai

# Inisialisasi API KEY

In [2]:
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
client = genai.Client()

# Prompt Engineering

In [3]:
print("Meminta Gemini untuk membuat dataset sintetik... (Mohon tunggu sekitar 10-20 detik)")

prompt = """
Kamu adalah asisten pembuat dataset Machine Learning.
Buatkan saya dataset sintetik sebanyak 500 baris. 

Skenario: Laporan kerusakan aset pabrik yang berisi cerita sebab-akibat antara keluhan orang awam dan laporan perbaikan oleh teknisi.

ATURAN KETAT:
- Pisahkan antar kolom HANYA menggunakan simbol pipe (|). JANGAN gunakan koma sebagai pemisah kolom.
- Jangan berikan teks pembuka atau penutup apapun. LANGSUNG berikan data mentahnya.

KOLOM YANG WAJIB ADA (Tepat 6 kolom):
teks_keluhan_awam|teks_laporan_teknisi|kategori_aset|severity|root_cause|tindakan

Pilihan untuk 4 kolom terakhir:
- kategori_aset: (HVAC, Pompa Air, Kelistrikan, Mesin Produksi)
- severity: (Rendah, Sedang, Tinggi)
- root_cause: (Tersumbat, Keausan, Overheat, Usia Pakai, Konsleting)
- tindakan: (Pembersihan, Penggantian Part, Pelumasan, Kalibrasi, Perbaikan Jaringan)

Contoh Baris yang Benar:
AC di ruang meeting lantai 2 netes air terus nih|udah disemprot selang pembuangannya karena mampet lumut, sekalian tambah freon|HVAC|Sedang|Tersumbat|Pembersihan
"""

Meminta Gemini untuk membuat dataset sintetik... (Mohon tunggu sekitar 10-20 detik)


# Memanggil Model Gemini

In [4]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
)

# Processing ke CSV

In [5]:
import os # Pastikan os sudah di-import

csv_raw_data = response.text

# 4. Membersihkan format markdown jika ada
if csv_raw_data.startswith("```"):
    # Membuang baris pertama (```text) dan baris terakhir (```)
    lines = csv_raw_data.strip().split('\n')
    csv_raw_data = '\n'.join(lines[1:-1])

# 5. Membaca data dengan pemisah | (pipe)
try:
    # Mengubah teks dari Gemini menjadi DataFrame
    df_sintetik = pd.read_csv(io.StringIO(csv_raw_data.strip()), sep='|')
    
    # 6. Menyimpan dataset ke file CSV lokal
    file_name = "../../data/synthetic_report_dataset.csv"
    
    # CEK: Apakah file sudah ada sebelumnya?
    if os.path.exists(file_name):
        # Jika sudah ada, mode='a' (append) dan header=False agar nama kolom tidak ikut masuk ke tengah data
        df_sintetik.to_csv(file_name, mode='a', header=False, index=False)
        print(f"\n✅ BERHASIL! Data baru ditambahkan (Append) ke '{file_name}'")
        
        # Mengecek total baris sekarang
        df_total = pd.read_csv(file_name)
        print(f"Total baris keseluruhan saat ini: {df_total.shape[0]}")
    else:
        # Jika file belum ada, buat file baru (overwrite biasa)
        df_sintetik.to_csv(file_name, index=False)
        print(f"\n✅ BERHASIL! Dataset awal berhasil dibuat dan disimpan sebagai '{file_name}'")
        print(f"Jumlah baris: {df_sintetik.shape[0]}")
    
    print("\n--- 5 Baris Data Terbaru ---")
    display(df_sintetik.head())
    
except Exception as e:
    print(f"❌ Gagal memproses format data. Error: {e}")
    print("Teks mentah dari Gemini:\n", csv_raw_data)


✅ BERHASIL! Data baru ditambahkan (Append) ke '../../data/synthetic_report_dataset.csv'
Total baris keseluruhan saat ini: 665

--- 5 Baris Data Terbaru ---


,AC di ruang meeting lantai 2 netes air terus nih,"Saluran pembuangan unit AC mampet karena lumut dan debu, sudah disemprot dan dibersihkan.",HVAC,Sedang,Tersumbat,Pembersihan
0,Pompa air utama mengeluarkan suara berisik dan...,"Bearing motor pompa aus dan berisik, sudah dig...",Pompa Air,Tinggi,Keausan,Penggantian Part
1,Lampu di gudang mati total,MCB di panel distribusi sudah habis usia pakai...,Kelistrikan,Sedang,Usia Pakai,Penggantian Part
2,"Mesin pengemas macet, produk tidak bisa lewat","Saluran feed material mesin tersumbat residu, ...",Mesin Produksi,Sedang,Tersumbat,Pembersihan
3,AC di lab kimia tiba-tiba mati total dan ada b...,Terjadi hubungan arus pendek pada kabel power ...,HVAC,Tinggi,Konsleting,Perbaikan Jaringan
4,Motor pompa air utama sangat panas dan mati se...,Motor pompa mengalami overheat karena beban be...,Pompa Air,Tinggi,Overheat,Penggantian Part
